# Exploratory data analysis: WeLoveReviews

This section gets to know the 500 written reviews before any modeling. It covers the data's structure, the
ratings, the review text, a few things worth checking, and whether any cleaning is needed.

First, let's load the file and see what we have.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

BLUE, INK, MUTED, GRID = "#2a78d6", "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": GRID,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": INK, "axes.titlelocation": "left",
    "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.bottom": False, "ytick.left": False, "font.size": 10,
})
pd.set_option("display.max_colwidth", None)

DATA_PATH = Path("../data/raw/reviews.csv")
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## Structure and missing values

There are three columns, as expected. Next we check their data types and whether anything is missing or out of range.

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "blank_strings": [(df[c].astype(str).str.strip() == "").sum() for c in df.columns],
    "unique_values": df.nunique(),
})
display(summary)
print("Duplicate review_id values:", df["review_id"].duplicated().sum())
print("Ratings outside 1-5:", (~df["rating"].between(1, 5)).sum())

## Rating distribution

The structure is sound, so now to the number behind the business question: does this file really average about 4.5 stars?

In [ ]:
rating_counts = df["rating"].value_counts().sort_index()
rating_table = pd.DataFrame({
    "reviews": rating_counts,
    "share_%": (rating_counts / len(df) * 100).round(1),
})
display(rating_table)

rating_band = df["rating"].map({1: "Negative", 2: "Negative", 3: "Neutral", 4: "Positive", 5: "Positive"})
band_share = (rating_band.value_counts(normalize=True) * 100).round(1).reindex(["Positive", "Neutral", "Negative"])

print(f"Mean rating: {df['rating'].mean():.2f}   Median rating: {df['rating'].median():.0f}")
print("Share by rating band (from stars only):")
print(band_share.to_string())

fig, ax = plt.subplots(figsize=(7, 3.2))
bars = ax.bar(rating_counts.index.astype(str), rating_counts.values, color=BLUE, width=0.6)
ax.bar_label(bars, padding=3, color=INK)
ax.set_title("Most reviews give 5 stars")
ax.set_xlabel("Star rating")
ax.set_ylabel("Number of reviews")
ax.yaxis.grid(True, color=GRID)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## Review text length

The ratings confirm the average. Now the text itself: how long are the reviews, and does length change with the rating?

In [ ]:
df["char_count"] = df["review_text"].str.len()
df["word_count"] = df["review_text"].str.split().str.len()

display(df[["char_count", "word_count"]].describe().round(1))
display(df.groupby("rating")[["char_count", "word_count"]].mean().round(1))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(df["word_count"], bins=range(df["word_count"].min(), df["word_count"].max() + 2),
        color=BLUE, edgecolor="#fcfcfb", linewidth=1)
ax.set_title("Reviews are short and similar in length")
ax.set_xlabel("Words per review")
ax.set_ylabel("Number of reviews")
ax.yaxis.grid(True, color=GRID)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## A small sample of reviews

Numbers only go so far, so let's read some. This sample takes two reviews per star rating, so the rare low ratings
show up too.

In [ ]:
sample = df.groupby("rating").sample(n=2, random_state=42)[["review_id", "rating", "review_text"]]
sample.style.hide(axis="index").set_properties(subset=["review_text"], **{"text-align": "left"})

## How repetitive is the text?

Several phrases recur in that sample. This check measures that, along with exact duplicates and how often the review
names the business.

In [ ]:
def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]


sentences = df["review_text"].map(split_sentences).explode()

duplicate_mask = df["review_text"].duplicated(keep=False)
print("Reviews whose text matches another review:", duplicate_mask.sum())
print("Extra copies beyond the first:", df["review_text"].duplicated().sum())
print("Rows with leading or trailing whitespace:", (df["review_text"] != df["review_text"].str.strip()).sum())
print("Total sentences:", len(sentences))
print("Distinct sentences:", sentences.nunique())
print("Reviews naming Harbor House Café:", df["review_text"].str.contains("Harbor House Café").sum())
print()
print("Most common sentences:")
print(sentences.value_counts().head(8).to_string())

## Do the stars and the words always agree?

This last check looks for mixed signals: reviews with high stars but lukewarm or critical wording, and the reverse.
It only counts wording, using no model, to show where the text and the rating may later pull apart.

In [ ]:
long_df = df[["review_id", "rating", "review_text"]].assign(sentence=df["review_text"].map(split_sentences)).explode("sentence")
sentence_groups = long_df.assign(group=long_df["rating"].map(lambda r: "high_4_5" if r >= 4 else "low_1_3"))

by_sentence = sentence_groups.pivot_table(index="sentence", columns="group", values="review_id", aggfunc="count", fill_value=0)
shared = by_sentence[(by_sentence["high_4_5"] > 0) & (by_sentence["low_1_3"] > 0)]
shared = shared[~shared.index.str.contains("Harbor House")].sort_values("high_4_5", ascending=False)
print("Non-opening sentences used in both high-star (4-5) and low-star (1-3) reviews:")
display(shared)

has_but = df["review_text"].str.contains(r"\bbut\b", case=False)
high = df["rating"] >= 4
print(f"4-5 star reviews containing 'but': {(has_but & high).sum()} of {high.sum()}")
print(f"  of which 4-star: {(has_but & (df['rating'] == 4)).sum()} of {(df['rating'] == 4).sum()}")
print(f"  of which 5-star: {(has_but & (df['rating'] == 5)).sum()} of {(df['rating'] == 5).sum()}")

lukewarm = shared.index.tolist()
has_lukewarm = df["review_text"].map(lambda t: any(s in lukewarm for s in split_sentences(t)))
print(f"4-5 star reviews containing one of those shared sentences: {(has_lukewarm & high).sum()}")

examples = df.loc[(has_but | has_lukewarm) & high, ["review_id", "rating", "review_text"]].head(6)
examples.style.hide(axis="index").set_properties(subset=["review_text"], **{"text-align": "left"})

## Key insights

_(pending)_

## Cleaning proposal

_(pending)_